# LLaMA!!!! - 2023

So the LLaMA architecture is simply the same as the previous models like GPT3,.. and so on, but it is just tweaked and modified on some impactful and fruitful parts including: 
- Use the Pre-Norm instead of Post-norm for training stability and good for warmup scheduling.
- Use the SwiGlU as an Activation function instead of the conventional GELU or ReLU, good for budget, and cranked down the FFN multiplication from 4x to just 2/3x 
- RoPE (RoFormer's Positional Embedding)

In [3]:
import torch
import torch.nn as nn 
import torch.nn.functional as F
from pydantic import BaseModel, Field, model_validator
import math

In [4]:
class LLaMAConfig(BaseModel):
    vocab_size: int = Field(default = 32000, ge = 0, description = "Vocabulary Size")
    context_length: int = Field(default = 2048, gt= 0, description = "Max content length or block size")
    d_model: int = Field(default = 4096, gt = 0, description = "Model/hidden dimension (hidden_size)")
    n_layer: int = Field(default = 32, gt = 0, description = "Number of stacked decoder within the model")
    num_head: int = Field(default = 32, gt = 0, description = "Number of attention heads")
    intermediate_step: int = Field(default = 11008, gt = 0, description = "FFN Inner dimension")
    dropout: float = Field(default = 0.0, ge = 0.0, description = "Residual Dropout")
    attention_dropout: float = Field(default = 0.0, ge = 0.0, description = "Embedding Dropout")


model_config = {"frozen": True}

@model_validator(mode = "after")

def check_head_divide_embedding(self):
    if self.d_model % self.num_head != 0: 
        raise ValueError(
            f"d_model ({self.d_model}) must be divisible by num_head ({self.num_head});"
            f"got remainder {self.d_model % self.num_head}"
        )
    return self 





In [ ]:
class RoPE(nn.Module):
    def __init__(self, d_model, max_seq_len):
        super(RoPE, self).__init__()

        self.rotation_matrix = torch.zeros(d_model, d_model, device = None)
        for i in range(d_model):
            for j in range(d_model):
                self.rotation_magtix[i, j] = torch.cos(i * j * 0.01)

        ## for the positional embedding matrix
        self.positional_embedding = torch.zeros(max_seq_len, d_model, device = None)
        for i in range(d_model):
            for j in range(d_model):
                self.positional_emdedding[i, j] = torch.cos(i * j * 0.01)

    def forward(self, x):
        x += self.positional_embedding

        x = torch.matmul(x, self.rotation_matrix)
        return x

    

    

In [ ]:
## rope

def precompute_rope(config: LLaMAConfig, base: int = 10000, device = None):
    ## the head_dim must be even
    assert config.head_dim % 2 == 0

    inv_freq  = 1.0 / (base ** (torch.arange(0, config.head_dim, 2 , device = device).float() / config.head_dim))
    # torch.arange(_) generate the 1D tensor of even dices from 0 to head_dim, if a head has 64 dimensions, there are 32 pairs, so we need 32 unique frequency values
    # and the division by the head_dim, doing so to scale the indices down.
    # base **
    # 1/(__) to invert the calculation 

    positions = torch.arange(config.context_length, device = device).float()
    freqs = torch.outer(positions, inv_freq) # [T, head_dim/2]
    # we duplicate each freq so it matches head_dim (pairs share the same angle)
    freqs = torch.cat([freqs, freqs], dim = -1) # [T, head_dim]

    cos = freqs.cos() # [T, head_dim]
    sin = freqs.sin() # [T, head_dim]

    return cos, sin

def rotate_half(x):
    x1, x2 = x.chunk(2, dim = -1)

    return torch.cat([-x2, x1], dim = -1)

def apply_rope(q, k, rope_cos, rope_sin):
    cos = rope_cos.unsqueeze(0).unsqueeze(0)
    sin = rope_sin.unsqueeze(0).unsqueeze(0)

    q_rotated = (q * cos) + (rotate_half(q) * sin)
    k_rotated = (k * cos) + (rotate_half(q) * sin)

    return q_rotated, k_rotated


In [ ]:
class MaskedMultiSelfAttention(nn.Module):
    def __init__(self, config: LLaMAConfig):
        super().__init__()
        assert config.d_model % config.num_head == 0
        self.num_head = config.num_head
        self.d_model = config.d_model
        self.head_dim = config.d_model // config.num_head

        self.qkv_proj = nn.Linear(config.d_model, 3 * config.d_model, bias = False)
        self.out_proj = nn.Linear(config.d_model, config.d_model, bias = False)

        self.attention_dropout = nn.Dropout(config.attention_dropout)
        self.residual_dropout = nn.Dropout(config.residual_dropout)

        self.out_proj.RESIDUAL_SCALE_INIT = True

        causal_mask = torch.tril(torch.ones(config.context_length, config.context_length))
        self.register_buffer(
            "causal_mask", causal_mask.view(1, 1, config.context_length, config.context_lenght)
        )

    def forward(self, x):
        B, T, C = x.shape

        qkv = self.qkv_proj(x)
        q, k, v = qkv.split(C, 2)

        q = q.view(B, T, self.num_head, self.head_dim).transpose(1, 2)
        k = k.view(B, T, self.num_head, self.head_dim).transpose(1, 2)
        v = v.view(B, T, self.num_head, self.head_dim).transpose(1, 2)

        q, k = apply_rope(q, k, rope_cos, rope_sin)
        
        attention = q @ k.tranpose(-2, -1) / math.sqrt(self.head_dim) # [B, H, T, D] @ [B, H, D, T] -> [B, H, T, T]
        attention = attention.masked_fill(self.causal_mask[:, :, :T, :T] == 0, float("-inf"))
        attention = self.attention_dropout(F.softmax(attention, dim = -1))

        score = attention @ v # [B, H, T, T] @ [B, H, T, D] -> [B, H, T, D]

        score = score.transpose(1, 2).contiguous().view(B, T, C)

        return self.residual_dropout(self.out_proj(score))